# Lab 27-04 · Ears, Glasses and Headsets

Companion to [27-04 · Wearables II](../../chapters/27-04-wearables-earbuds-and-glasses.md). Each part asks you to **predict** first, then run the cell. All parameters are the chapter's assumptions.

In [1]:
import math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import headworn as hw

## Part 1 · The ANC delay limit

*Predict* the frequency at which a 50 µs delay leaves 10 dB of cancellation.

In [2]:
tau = 50e-6
f10 = math.asin(10**(-10/20)/2) / (math.pi*tau)
print(f'10 dB of cancellation at {f10/1000:.2f} kHz with a 50 us delay')

f = np.logspace(1, np.log10(5000), 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
for d in (20e-6, 50e-6, 100e-6, 1e-3):
    ax.semilogx(f, [hw.anc_attenuation_db(x, d) for x in f], label=f'{d*1e6:.0f} us')
ax.axhline(0, color='k', lw=0.8); ax.set_ylim(-10, 45)
ax.set_xlabel('frequency (Hz)'); ax.set_ylabel('cancellation (dB)'); ax.legend(); ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()

10 dB of cancellation at 1.01 kHz with a 50 us delay


## Part 2 · LE Audio bit rate

*Predict* the radio current at 192 kb/s in 10 ms frames (Example 27-04.4's radio).

In [3]:
def current(kbps, frame_ms):
    n = hw.lc3_frame_octets(kbps, frame_ms)
    return hw.stream_current_ua(hw.le_airtime_us(n, 2), hw.le_airtime_us(0, 2), 150, 200, 3.6, 4.1, 1.2, frame_ms)

print(f'192 kb/s, 10 ms: {hw.lc3_frame_octets(192, 10):.0f} octets, {current(192, 10):.0f} uA')
rates = np.arange(32, 193, 8)
fig, ax = plt.subplots(figsize=(6, 3.5))
for fr in (7.5, 10):
    ax.plot(rates, [current(r, fr) for r in rates], label=f'{fr} ms frames')
ax.set_xlabel('LC3 bit rate (kb/s)'); ax.set_ylabel('radio current (uA)'); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()

192 kb/s, 10 ms: 240 octets, 463 uA


## Part 3 · Warm surface for a budget

*Predict* the area a pair of glasses needs to shed 0.5 W at an 8 K rise with h = 8 W/m²·K.

In [4]:
print(f'area for 0.5 W: {1e4*0.5/(8*8):.1f} cm^2')
area_cm2 = np.linspace(20, 120, 101)
fig, ax = plt.subplots(figsize=(6, 3.5))
for dT in (5, 8, 10):
    ax.plot(area_cm2, [hw.surface_power_w(8, a*1e-4, dT) for a in area_cm2], label=f'{dT} K rise')
ax.set_xlabel('warm surface (cm^2)'); ax.set_ylabel('sustained power (W)'); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()

area for 0.5 W: 78.1 cm^2


## Part 4 · Foveation

*Predict* the shading work for a 30° fovea in a 100° field with a quarter-scale periphery.

In [5]:
print(f'work: {hw.foveated_fraction(30, 100, 0.25):.3f}')
fov = np.linspace(5, 60, 56)
fig, ax = plt.subplots(figsize=(6, 3.5))
for s in (0.25, 1/3, 0.5):
    ax.plot(fov, [hw.foveated_fraction(x, 100, s) for x in fov], label=f'periphery scale {s:.2f}')
ax.set_xlabel('fovea size (deg)'); ax.set_ylabel('shading work (fraction)'); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()

work: 0.147


## Discussion

Compare your predictions with the expected results in the [lab README](README.md). For each device class, which of these four curves would you show a product manager, and why?